In [1]:
import os
import sys

os.chdir(r'C:\Users\mhuss\treasury-data-validator')
sys.path.insert(0, r'C:\Users\mhuss\treasury-data-validator')

import pandas as pd
import json

from src.db_connector import get_connection
from src.validators import (
    validate_completeness,
    validate_known_counterparties,
    validate_date_logic,
    validate_positive_notional,
    validate_counterparty_limits,
    validate_emir_reporting,
    validate_no_duplicates,
)
from src.report_generator import generate_report, save_report

print("Working directory:", os.getcwd())
print("Libraries loaded. Pipeline ready.")

Working directory: C:\Users\mhuss\treasury-data-validator
Libraries loaded. Pipeline ready.


In [2]:
base = r'C:\Users\mhuss\treasury-data-validator'

transactions = pd.read_csv(os.path.join(base, 'data', 'raw_transactions.csv'))
limits = pd.read_csv(os.path.join(base, 'data', 'counterparty_limits.csv'))

print(f"Transactions loaded: {transactions.shape[0]} rows, {transactions.shape[1]} columns")
print(f"Counterparties loaded: {limits.shape[0]} rows")
print()
print("Columns:", list(transactions.columns))

Transactions loaded: 103 rows, 16 columns
Counterparties loaded: 5 rows

Columns: ['trade_id', 'trade_date', 'value_date', 'maturity_date', 'instrument_type', 'base_currency', 'quote_currency', 'notional_amount', 'notional_currency', 'counterparty_id', 'buy_sell', 'status', 'trade_repository_ref', 'clearing_eligible', 'booking_timestamp', 'source_system']


In [3]:
print("=== DIRTY DATA OVERVIEW ===\n")

# Duplicates
dupes = transactions[transactions.duplicated('trade_id', keep=False)]
print(f"Duplicate trade_ids: {dupes['trade_id'].nunique()} trade IDs appear more than once")
print(dupes[['trade_id', 'booking_timestamp']].drop_duplicates().to_string(index=False))

print()

# Null counterparty
null_cp = transactions[transactions['counterparty_id'].isna() | (transactions['counterparty_id'] == '')]
print(f"Null counterparty_id: {len(null_cp)} records")
print(null_cp[['trade_id', 'counterparty_id']].to_string(index=False))

print()

# Invalid instrument types
valid_types = {'FX_FORWARD', 'INTEREST_RATE_SWAP', 'CASH_DEPOSIT', 'COMMODITY_FORWARD'}
invalid = transactions[~transactions['instrument_type'].str.strip().isin(valid_types)]
print(f"Invalid instrument types: {len(invalid)} records")
print(invalid[['trade_id', 'instrument_type']].to_string(index=False))

=== DIRTY DATA OVERVIEW ===

Duplicate trade_ids: 3 trade IDs appear more than once
trade_id   booking_timestamp
TRD-0011 2024-06-23T00:00:00
TRD-0026 2024-04-19T00:00:00
TRD-0041 2024-11-18T00:00:00
TRD-0011 2024-06-23T02:00:00
TRD-0026 2024-04-19T02:00:00
TRD-0041 2024-11-18T02:00:00

Null counterparty_id: 4 records
trade_id counterparty_id
TRD-0006             NaN
TRD-0016             NaN
TRD-0051             NaN
TRD-0071             NaN

Invalid instrument types: 2 records
trade_id instrument_type
TRD-0009         FX_SPOT
TRD-0023            BOND


In [4]:
# Stage 2: Clean inline
df_clean = transactions.copy()

# Strip whitespace from string fields
string_cols = ['trade_id', 'instrument_type', 'base_currency', 'quote_currency',
               'notional_currency', 'counterparty_id', 'buy_sell', 'status', 'source_system']
for col in string_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()
    df_clean[col] = df_clean[col].replace('nan', pd.NA)

# Standardize dates
date_cols = ['trade_date', 'value_date', 'maturity_date']
for col in date_cols:
    df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce').dt.strftime('%Y-%m-%d')

# Remove duplicates — keep latest booking_timestamp
df_clean['booking_timestamp'] = pd.to_datetime(df_clean['booking_timestamp'], errors='coerce')
df_clean = df_clean.sort_values('booking_timestamp', ascending=False)
df_clean = df_clean.drop_duplicates('trade_id', keep='first')

# Remove invalid instrument types
valid_types = {'FX_FORWARD', 'INTEREST_RATE_SWAP', 'CASH_DEPOSIT', 'COMMODITY_FORWARD'}
df_clean = df_clean[df_clean['instrument_type'].isin(valid_types)]

print(f"Records after cleaning: {len(df_clean)} (from {len(transactions)} raw)")

# Stage 3: Run all validations
conn = get_connection()
results = [
    validate_completeness(df_clean),
    validate_known_counterparties(df_clean, limits),
    validate_date_logic(df_clean),
    validate_positive_notional(df_clean),
    validate_counterparty_limits(df_clean, limits),
    validate_emir_reporting(df_clean),
    validate_no_duplicates(df_clean),
]

print("Validations complete.")

Records after cleaning: 98 (from 103 raw)
Validations complete.


In [5]:
print("=== VALIDATION SUMMARY ===\n")

passed = [r for r in results if r['status'] == 'PASS']
failed = [r for r in results if r['status'] == 'FAIL']

overall = 'PASS' if len(failed) == 0 else 'FAIL'
print(f"Overall status: {overall}")
print(f"Rules passed:   {len(passed)} / {len(results)}")
print(f"Rules failed:   {len(failed)} / {len(results)}")
print()

for r in results:
    icon = "PASS" if r['status'] == 'PASS' else "FAIL"
    print(f"{icon}  {r['rule_id']:8}  {r['category']:20}  passed={r['passed']}  failed={r['failed']}")

=== VALIDATION SUMMARY ===

Overall status: FAIL
Rules passed:   2 / 7
Rules failed:   5 / 7

FAIL  V1        COMPLETENESS          passed=94  failed=4
FAIL  V3        COMPLETENESS          passed=94  failed=4
FAIL  V4_V5     FINANCIAL_LOGIC       passed=95  failed=3
FAIL  V6        FINANCIAL_LOGIC       passed=96  failed=2
PASS  V10       LIMIT_MONITORING      passed=5  failed=0
FAIL  V13       REGULATORY            passed=10  failed=8
PASS  V15       REGULATORY            passed=98  failed=0


In [6]:
print("=== V13: EMIR COMPLIANCE FAILURES ===\n")
emir_result = next(r for r in results if r['rule_id'] == 'V13')
print(f"Active trades missing trade_repository_ref: {emir_result['failed']}")
print(f"Regulatory note: {emir_result.get('regulatory_note', 'Null trade_repository_ref on ACTIVE trades = EMIR breach')}")
print()
emir_df = pd.DataFrame(emir_result['failing_records'])
print(emir_df.to_string(index=False))

print()
print("=== V4_V5: DATE LOGIC FAILURES ===\n")
date_result = next(r for r in results if r['rule_id'] == 'V4_V5')
print(f"Records with impossible dates: {date_result['failed']}")
print()
date_df = pd.DataFrame(date_result['failing_records'])
print(date_df.to_string(index=False))

=== V13: EMIR COMPLIANCE FAILURES ===

Active trades missing trade_repository_ref: 8
Regulatory note: Null trade_repository_ref = EMIR reporting gap

trade_id trade_date    instrument_type
TRD-0040 2024-12-10         FX_FORWARD
TRD-0038 2024-12-06  COMMODITY_FORWARD
TRD-0018 2024-12-03         FX_FORWARD
TRD-0034 2024-05-22       CASH_DEPOSIT
TRD-0007 2024-05-18 INTEREST_RATE_SWAP
TRD-0033 2024-04-12 INTEREST_RATE_SWAP
TRD-0035 2024-02-25  COMMODITY_FORWARD
TRD-0004 2024-02-19         FX_FORWARD

=== V4_V5: DATE LOGIC FAILURES ===

Records with impossible dates: 3

trade_id trade_date value_date maturity_date
TRD-0008 2024-10-17 2024-10-14    2025-03-08
TRD-0021 2024-07-24 2024-07-21    2024-11-17
TRD-0036 2024-12-28 2025-01-02    2024-12-23


In [7]:
base = r'C:\Users\mhuss\treasury-data-validator'

report = generate_report(results)
save_report(report, os.path.join(base, 'outputs', 'validation_report.json'))

print(f"Overall: {report['report_metadata']['overall_status']}")
print(f"Rules failed: {report['report_metadata']['rules_failed']}")
print("Report saved to outputs/validation_report.json")

Report saved to: C:\Users\mhuss\treasury-data-validator\outputs\validation_report.json
Overall: FAIL
Rules failed: 5
Report saved to outputs/validation_report.json
